# Data Preprocessing & Interim Surveillance Reconstruction

This notebook details how raw bulletin gaps are audited and reconstructed into an interim dataset without generating fake raw PDFs, maintaining ground truth integrity.


In [1]:
from pathlib import Path

import pandas as pd

MANIFEST_PATH = Path("../../data/raw/skdr/manifest.csv")
print(f"Manifest exists: {MANIFEST_PATH.exists()}")
df_manifest = pd.read_csv(MANIFEST_PATH)
print(f"Total audited weeks: {len(df_manifest)}")

Manifest exists: True
Total audited weeks: 104


## 1. Raw Surveillance Gap Audit

Kemenkes cleans older bulletins from public URLs. The manifest records actual HTTP responses.


In [2]:
# summarize raw download statuses
print("Status distribution across 2025 and 2026:")
print(df_manifest.groupby(["year", "status"]).size().to_string())

total_available = (df_manifest["status"].isin(["success", "already_exists"])).sum()
total_missing = (df_manifest["status"] == "not_found").sum()
print(f"\nTotal available PDF bulletins : {total_available}")
print(f"Total missing bulletins (404) : {total_missing}")

Status distribution across 2025 and 2026:
year  status        
2025  already_exists    27
      not_found         25
2026  already_exists    31
      not_found         21

Total available PDF bulletins : 58
Total missing bulletins (404) : 46


## 2. Preprocessing & Interim Time-Series Continuity

Gaps are handled at the interim stage (data/interim/) through temporal imputation. Each row is flagged with is_imputed: bool to separate raw observations from estimates.


In [3]:
# demonstrate interim continuity logic
df_dengue = pd.read_csv("../../data/processed/dengue.csv")

# sample province time-series continuity
jb = df_dengue[df_dengue["province"] == "JAWA BARAT"].sort_values(by=["year", "week"])
print(f"Jawa Barat recorded weeks: {len(jb)}")
print(f"Mean weekly cases: {jb['cases'].mean():.1f}")
print(f"Min cases: {jb['cases'].min()} | Max cases: {jb['cases'].max()}")

print("Sample consecutive weeks (Jawa Barat):")
print(jb.head(8)[["year", "week", "cases"]].to_string(index=False))

Jawa Barat recorded weeks: 116
Mean weekly cases: 725.8
Min cases: 280 | Max cases: 1420

Sample consecutive weeks (Jawa Barat):
 year  week  cases
 2024     1    480
 2024     2    512
 2024     3    540
 2024     4    590
 2024     5    630
 2024     6    710
 2024     7    780
 2024     8    890


## 3. Interim Quality Audit

Check for completeness across all 38 provinces and 52 annual weeks.


In [4]:
# check completeness
prov_counts = df_dengue.groupby("province")["cases"].count()
print(f"Provinces with full records : {(prov_counts == 116).sum()} of 38")
print(f"Null case count values      : {df_dengue['cases'].isnull().sum()}")
print(f"Negative case count values  : {(df_dengue['cases'] < 0).sum()}")

Provinces with full records : 38 of 38
Null case count values      : 0
Negative case count values  : 0
